## Ontology Justification

Three ontologies are used for the semantic mapping of this dataset:

**schema.org** (`https://schema.org`) is used for general descriptive attributes
such as identifiers, names, dates, times, and counts. It is the most widely adopted
structured data vocabulary on the web, supported natively by Google, Microsoft, and
Yandex, making it the best choice for maximising interoperability of non-spatial,
non-numeric fields.

**GeoSPARQL** (`http://www.opengis.net/ont/geosparql#`) is used for all spatial
attributes (easting, northing, longitude, latitude, area). It is the official OGC
standard ontology for representing geographic information as linked data, and is the
most specific and properly curated ontology available for geospatial concepts.

**W3C WGS84 Geo Positioning** (`http://www.w3.org/2003/01/geo/wgs84_pos#`) is used
specifically for WGS84 longitude and latitude, as it provides the precise concepts
`wgs84_pos:long` and `wgs84_pos:lat` for decimal-degree geographic coordinates,
which are more semantically specific than the generic GeoSPARQL geometry terms.

In [6]:
import sys
!{sys.executable} -m pip install requests pandas dbrepo

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 2.9 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.1 MB 3.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.1 MB/s  0:00:00

   ----------------------------------------  0/12 [urllib3]
   ----------------------------------------  0/12 [urllib3]
   ----------------------------------------  0/12 [urllib3]
   --- ------------------------------------  1/12 [typing-extensions]
   ------ ---------------------------------  2/12 [pika]
   ------ ---------------------------------  2/12 [pika]
   ------ ---------------------------------  2/12 [pika]
   ---------- -----------------------------  3/12 [idna]
   ------------- --------------------------  4/12 [charset_normalizer]
  


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import json
import time
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
from dbrepo.RestClient import RestClient

ENDPOINT     = "https://test.dbrepo.tuwien.ac.at"
USERNAME     = "mehedy360"                                          # your DBRepo username
PASSWORD     = "Motorolla440$"                                  # your DBRepo password
DATABASE_ID  = "f36ef3e2-1aee-4526-b3ea-82f661a9261a"            # rta_stats19_north_yorkshire
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"            # MariaDB Galera 11.3.2

AUTH    = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
client  = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

print("Authenticated as:", client.whoami())

mehedy360
Authenticated as: mehedy360


In [19]:
import json
import requests
from requests.auth import HTTPBasicAuth
from dbrepo.RestClient import RestClient

# ── Config ────────────────────────────────────────────────────────────
ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = "mehedy360"
PASSWORD    = "Motorolla440$"
DATABASE_ID = "f36ef3e2-1aee-4526-b3ea-82f661a9261a"

AUTH    = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
client  = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)
print("Authenticated as:", client.whoami())

# ── Fetch live schema ─────────────────────────────────────────────────
r = requests.get(f"{ENDPOINT}/api/v1/database/{DATABASE_ID}",
                 auth=AUTH, headers=HEADERS)
print(f"Database fetch: HTTP {r.status_code}")
if r.status_code != 200:
    raise SystemExit(f"Cannot fetch database: {r.text[:300]}")

db = r.json()

# Build lookup maps
col_obj_map  = {}   # "table.col" -> full column object
table_id_map = {}   # "table_internal_name" -> table_id

for tbl in db["tables"]:
    tname = tbl["internal_name"]
    table_id_map[tname] = tbl["id"]
    for col in tbl["columns"]:
        col_obj_map[f"{tname}.{col['internal_name']}"] = col

print(f"Tables:  {len(table_id_map)}")
print(f"Columns: {len(col_obj_map)}")

# ── Concept URI mappings ──────────────────────────────────────────────
CONCEPT_URIS = {
    # accident (fact table)
    "accident.police_ref":              "https://schema.org/identifier",
    "accident.accident_date":           "https://schema.org/startDate",
    "accident.accident_time":           "https://schema.org/startTime",
    "accident.day_of_week":             "https://schema.org/dayOfWeek",
    "accident.easting":                 "http://www.opengis.net/ont/geosparql#asWKT",
    "accident.northing":                "http://www.opengis.net/ont/geosparql#asWKT",
    "accident.longitude":               "http://www.w3.org/2003/01/geo/wgs84_pos#long",
    "accident.latitude":                "http://www.w3.org/2003/01/geo/wgs84_pos#lat",
    "accident.severity_id":             "https://schema.org/value",
    "accident.road_cond_id":            "https://schema.org/value",
    "accident.light_condition_id":      "https://schema.org/value",
    "accident.weather_condition_id":    "https://schema.org/value",
    "accident.special_condition_id":    "https://schema.org/value",
    "accident.carriageway_hazard_id":   "https://schema.org/value",
    "accident.casualties":              "https://schema.org/numberOfItems",
    "accident.vehicles":                "https://schema.org/numberOfItems",
    "accident.road_type_id":            "https://schema.org/value",
    "accident.speed_limit_mph":         "https://schema.org/speed",
    "accident.junction_detail_id":      "https://schema.org/value",
    "accident.junction_control_id":     "https://schema.org/value",
    "accident.crossing_control_id":     "https://schema.org/value",
    "accident.crossing_facility_id":    "https://schema.org/value",
    "accident.force_id":                "https://schema.org/value",
    "accident.oa11_code":               "http://www.opengis.net/ont/geosparql#sfContains",
    "accident.reporting_authority_id":  "https://schema.org/value",
    # accident_road
    "accident_road.police_ref":         "https://schema.org/identifier",
    "accident_road.road_sequence":      "https://schema.org/position",
    "accident_road.road_class_id":      "https://schema.org/value",
    "accident_road.road_number":        "https://schema.org/identifier",
    # output_area
    "output_area.oa11_code":            "https://schema.org/identifier",
    "output_area.lsoa_id":              "https://schema.org/identifier",
    "output_area.area_hectares":        "http://www.opengis.net/ont/geosparql#hasArea",
    "output_area.rurality_code":        "https://schema.org/value",
    "output_area.rurality_description": "https://schema.org/description",
    "output_area.rural_urban":          "https://schema.org/description",
    # lower_super_output_area
    "lower_super_output_area.lsoa_id":    "https://schema.org/identifier",
    "lower_super_output_area.lsoa_name":  "https://schema.org/name",
    "lower_super_output_area.lad12_code": "https://schema.org/identifier",
    # local_authority_district
    "local_authority_district.lad12_code":     "https://schema.org/identifier",
    "local_authority_district.lad12_code_old": "https://schema.org/identifier",
    "local_authority_district.lad_name":       "https://schema.org/name",
    # police_force
    "police_force.force_id":   "https://schema.org/identifier",
    "police_force.force_name": "https://schema.org/name",
    # lookup tables
    "severity_type.severity_id":                   "https://schema.org/identifier",
    "severity_type.description":                   "https://schema.org/name",
    "road_surface_condition.condition_id":         "https://schema.org/identifier",
    "road_surface_condition.description":          "https://schema.org/name",
    "light_condition.condition_id":                "https://schema.org/identifier",
    "light_condition.description":                 "https://schema.org/name",
    "weather_condition.condition_id":              "https://schema.org/identifier",
    "weather_condition.description":               "https://schema.org/name",
    "special_condition_at_site.condition_id":      "https://schema.org/identifier",
    "special_condition_at_site.description":       "https://schema.org/name",
    "carriageway_hazard.hazard_id":                "https://schema.org/identifier",
    "carriageway_hazard.description":              "https://schema.org/name",
    "road_type.type_id":                           "https://schema.org/identifier",
    "road_type.description":                       "https://schema.org/name",
    "road_class.class_id":                         "https://schema.org/identifier",
    "road_class.description":                      "https://schema.org/name",
    "junction_detail.detail_id":                   "https://schema.org/identifier",
    "junction_detail.description":                 "https://schema.org/name",
    "junction_control.control_id":                 "https://schema.org/identifier",
    "junction_control.description":                "https://schema.org/name",
    "pedestrian_crossing_control.control_id":      "https://schema.org/identifier",
    "pedestrian_crossing_control.description":     "https://schema.org/name",
    "pedestrian_crossing_facility.facility_id":    "https://schema.org/identifier",
    "pedestrian_crossing_facility.description":    "https://schema.org/name",
    "reporting_authority.authority_id":            "https://schema.org/identifier",
    "reporting_authority.description":             "https://schema.org/name",
}

# ── Apply mappings via update_table_column ────────────────────────────
ok_count, fail_count, skip_count = 0, 0, 0

for col_key, concept_uri in CONCEPT_URIS.items():
    if col_key not in col_obj_map:
        print(f"  SKIP  {col_key:50s}  (not in schema)")
        skip_count += 1
        continue

    col       = col_obj_map[col_key]
    table_name = col_key.split(".")[0]
    table_id  = table_id_map[table_name]

    try:
        client.update_table_column(
            database_id = DATABASE_ID,
            table_id    = table_id,
            column_id   = col["id"],
            concept_uri = concept_uri,
        )
        print(f"  ok    {col_key:50s}  → {concept_uri}")
        ok_count += 1
    except Exception as e:
        err = str(e)[:150]
        print(f"  FAIL  {col_key:50s}  {err}")
        fail_count += 1

print(f"\nDone — ok={ok_count}  fail={fail_count}  skip={skip_count}")

mehedy360
Authenticated as: mehedy360
Database fetch: HTTP 200
Tables:  19
Columns: 69
  ok    accident.police_ref                                 → https://schema.org/identifier
  ok    accident.accident_date                              → https://schema.org/startDate
  ok    accident.accident_time                              → https://schema.org/startTime
  ok    accident.day_of_week                                → https://schema.org/dayOfWeek
  ok    accident.easting                                    → http://www.opengis.net/ont/geosparql#asWKT
  ok    accident.northing                                   → http://www.opengis.net/ont/geosparql#asWKT
  ok    accident.longitude                                  → http://www.w3.org/2003/01/geo/wgs84_pos#long
  ok    accident.latitude                                   → http://www.w3.org/2003/01/geo/wgs84_pos#lat
  ok    accident.severity_id                                → https://schema.org/value
  ok    accident.road_cond_id         